In [ ]:
import numpy as np
from scipy.stats import norm, qmc
import pandas as pd
from joblib import Parallel, delayed
import multiprocessing
import os

# ============================================================
# Prelec probability distortion
# ============================================================

def prelec_weight(p, alpha):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return np.exp(-(-np.log(p)) ** alpha)


# ============================================================
# Posterior threat probability
# ============================================================

def posterior_threat(x, pi_hat, params):

    f1 = norm.pdf(x, params["mu1"], params["sigma"])
    f0 = norm.pdf(x, params["mu0"], params["sigma"])

    return (f1 * pi_hat) / (f1 * pi_hat + f0 * (1 - pi_hat))


# ============================================================
# Simulation of many lives
# ============================================================

def simulate_many(alpha, threats, cues, params):

    n_lives = threats.shape[0]
    T = params["T"]

    # belief parameters
    a_pi = np.full(n_lives, params["pi_true"] * params["prior_strength"])
    b_pi = np.full(n_lives, (1 - params["pi_true"]) * params["prior_strength"])

    fitness_history = np.zeros((n_lives, T))

    fitness_L = params["fitness_L"]
    fitness_F = params["fitness_F"]

    for t in range(T):

        pi_hat = a_pi / (a_pi + b_pi)

        x = cues[:, t]
        threat = threats[:, t]

        q = posterior_threat(x, pi_hat, params)

        # distorted probability
        q_d = prelec_weight(q, alpha)

        # Expected survival if stay
        expected_survival_stay = q_d * fitness_L + (1 - q_d) * 1.0

        # Expected survival if flee
        expected_survival_flee = q_d * 1.0 + (1 - q_d) * fitness_F

        flee = expected_survival_flee > expected_survival_stay

        # Realized fitness
        fitness = np.where(
            flee,
            np.where(threat, 1.0, fitness_F),
            np.where(threat, fitness_L, 1.0)
        )

        fitness_history[:, t] = fitness

        # learning occurs only when staying
        observe = ~flee

        a_pi += observe * threat.astype(float)
        b_pi += observe * (1 - threat.astype(float))

    # geometric mean survival
    geom = np.exp(np.mean(np.log(fitness_history), axis=1))

    return np.mean(geom)


# ============================================================
# Alpha optimization
# ============================================================

def compute_weighted_alpha(params, seed=None):

    if seed is not None:
        np.random.seed(seed)

    n_lives = params["n_lives"]
    T = params["T"]

    threats = np.random.rand(n_lives, T) < params["pi_true"]

    cues = np.where(
        threats,
        np.random.normal(params["mu1"], params["sigma"], (n_lives, T)),
        np.random.normal(params["mu0"], params["sigma"], (n_lives, T))
    )

    alpha_grid = np.logspace(-0.7, 0.7, 200)

    fitness_values = np.array([
        simulate_many(alpha, threats, cues, params)
        for alpha in alpha_grid
    ])

    best_index = np.argmax(fitness_values)

    return alpha_grid[best_index], fitness_values[best_index]


# ============================================================
# LHS Experiment
# ============================================================

N_SAMPLES = 5000
OUTPUT_FILE = "/kaggle/working/lhs_alpha_results.csv"
N_JOBS = max(1, multiprocessing.cpu_count() - 1)

bounds = {
    "pi_true": (0.01, 0.3),
    "fitness_L": (0.05, 0.75),
    "fitness_F": (0.9, 0.999),
    "sigma": (0.2, 1.5),
    "T": (5, 50),
    "prior_strength": (0.5, 10.0),
}

param_names = list(bounds.keys())
dim = len(param_names)

sampler = qmc.LatinHypercube(d=dim)
unit_samples = sampler.random(n=N_SAMPLES)


# ============================================================
# Transform LHS samples into parameter values
# ============================================================

def transform_sample(sample_row):

    params = {}

    for i, name in enumerate(param_names):

        low, high = bounds[name]
        u = sample_row[i]

        if name == "T":
            val = int(low + u * (high - low))
        else:
            val = low + u * (high - low)

        params[name] = val

    return params


param_list = [transform_sample(row) for row in unit_samples]


# ============================================================
# Evaluate parameter set
# ============================================================

def evaluate_param_set(p, row_index):

    local_params = {
        **p,
        "n_lives": 10000,
        "mu0": 0.0,
        "mu1": 1.0,
    }

    try:
        alpha_weighted, fitness = compute_weighted_alpha(local_params, seed=row_index)

    except Exception as e:
        print("Error:", e)
        alpha_weighted, fitness = np.nan, np.nan

    return {**p, "alpha_weighted": alpha_weighted, "fitness": fitness}


# ============================================================
# Run experiment
# ============================================================

print(f"Running {N_SAMPLES} LHS samples using {N_JOBS} cores...")

results = Parallel(n_jobs=N_JOBS)(
    delayed(evaluate_param_set)(p, idx)
    for idx, p in enumerate(param_list)
)

df = pd.DataFrame(results)

df.to_csv(OUTPUT_FILE, index=False)

print(f"Saved results to {OUTPUT_FILE}")
print(os.listdir("/kaggle/working"))